# 🎓 Tutorial RAG Didático: Fases 4 e 5 (Avaliando com DeepEval)

Com nossas Fases 1 a 3 construídas e instrumentadas pelo Langfuse (agora rodando liso na arquitetura OpenTelemetry v4), avançamos para o coração analítico deste tutorial: **Avaliar a Qualidade do nosso RAG de forma automatizada.**

Nesta fase, integraremos o *DeepEval*, um framework de LLM-as-a-judge (LLM como juiz) para quantificar as métricas de resposta e recuperação que discutimos na literatura teórica.

---
## ⚖️ Fase 4: Avaliação Sistêmica Baseada no DeepEval
O DeepEval consome dados em formato de Test Cases (`LLMTestCase`). Precisamos fornecer a ele exatamente o quê o usuário perguntou, as evidências que achamos, a resposta que demos e a resposta ideal.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# Se você não rodou o notebook anterior nesta mesma sessão, precisamos das dependências base
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric
)
from deepeval import evaluate

# Recapitulando nossos dados da Fase 2
exemplo_input = "Tomar Ibuprofeno e aspirina no mesmo dia pode desencadear problemas?"
exemplo_expected_output = "Sim. A combinação agrava a irritação do estômago e acentua significativamente riscos de hemorragia e ulceração."

# Vamos simular que o output e as evidências vieram direta do nosso RAG pipeline (Fase 3)
output_gerado_pelo_llm = "Sim, o ibuprofeno em conjunto com a aspirina é categorizado como uma formulação de AINEs em dobro. Isso agrava riscos de hemorragias gástricas severely."
evidencias_recuperadas_pelo_chromadb = [
    "Contraindicações de AINEs: Tanto o Ibuprofeno quanto o Ácido Acetilsalicílico (Aspirina) enquadram-se na extensa classe dos Anti-inflamatórios Não Esteroidais. A ingestão sistêmica ou combinada potencializa massivamente as irritações da mucosa gástrica, aumentando progressivamente os riscos cirúrgicos de hemorragia ou surgimento de úlceras duodenais extremas."
]

print("✅ Componentes da avaliação engatados!")

✅ Componentes da avaliação engatados!


### 4.1 Instanciar o Caso de Teste
A infraestrutura de teste demanda mapeamento claro.

In [2]:
caso_de_teste_didatico = LLMTestCase(
    input=exemplo_input,
    actual_output=output_gerado_pelo_llm,
    expected_output=exemplo_expected_output,
    retrieval_context=evidencias_recuperadas_pelo_chromadb
)


### 4.2 Métricas de Geração (Generation Metrics)
Neste estágio, verificamos a competência do LLM que *gera* a resposta (ex: GPT-4o-mini). 

1. **Answer Relevancy:** A resposta omitiu tópicos cruciais da pergunta ou enrolou? (Não liga para fatos, apenas para relevância do texto contra a pergunta).
2. **Faithfulness (Fidelidade/Alucinação):** A resposta inventou dados que as evidências não confirmam? (Não liga para a pergunta inicial, cruza *Output* vis-a-vis *Context*).

In [3]:
# Iniciando as métricas com modelo GPT-4o como juiz estrito
metrica_relevancia = AnswerRelevancyMetric(threshold=0.7, model="gpt-4o")
metrica_fidelidade = FaithfulnessMetric(threshold=0.7, model="gpt-4o")

print("⚖️ Iniciando julgamento da Geração...")
metrica_relevancia.measure(caso_de_teste_didatico)
metrica_fidelidade.measure(caso_de_teste_didatico)

print(f"\nRelevância da Resposta: {metrica_relevancia.score} (Passou: {metrica_relevancia.is_successful})")
print(f"  > Razão: {metrica_relevancia.reason}")

print(f"\nFidelidade (Sem Alucinação): {metrica_fidelidade.score} (Passou: {metrica_fidelidade.is_successful})")
print(f"  > Razão: {metrica_fidelidade.reason}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

⚖️ Iniciando julgamento da Geração...



Relevância da Resposta: 1.0 (Passou: <bound method AnswerRelevancyMetric.is_successful of <deepeval.metrics.answer_relevancy.answer_relevancy.AnswerRelevancyMetric object at 0x110b186e0>>)
  > Razão: The score is 1.00 because the response perfectly addresses the question about taking Ibuprofen and aspirin on the same day without any irrelevant information. Great job on staying focused and relevant!

Fidelidade (Sem Alucinação): 1.0 (Passou: <bound method FaithfulnessMetric.is_successful of <deepeval.metrics.faithfulness.faithfulness.FaithfulnessMetric object at 0x110b196a0>>)
  > Razão: The score is 1.00 because there are no contradictions, indicating that the actual output is perfectly aligned with the retrieval context. Great job maintaining accuracy and consistency!


### 4.3 Métricas de Recuperação (Context Metrics)
Neste estágio, testamos se o banco vetorial + Chunking (nosso *Retriever*) foi bom. O gerador nem participa.

1. **Contextual Recall:** Dentre a lista de informações que a "Resposta Ideal" precisa ter, seu contexto conseguiu puxar do PDF as frases que cobrem tudo? 
2. **Contextual Precision:** Supondo que você achou as informações corretas (Recall), elas vieram no Ranking mais alto do vetor ou estavam em útilmo misturadas com "lixo"?
3. **Contextual Relevancy:** Quão limpo é seu contexto? Se você entrega 1 página para o LLM e 1 parágrafo útil responde a questão, 90% do seu contexto é ruído irrelevante que só custa token e dinheiro.

In [4]:
metrica_recall = ContextualRecallMetric(threshold=0.7, model="gpt-4o")
metrica_precisao = ContextualPrecisionMetric(threshold=0.7, model="gpt-4o")
metrica_relev_contexto = ContextualRelevancyMetric(threshold=0.7, model="gpt-4o")

print("⚖️ Iniciando julgamento do Contexto/Retriever...")
metrica_recall.measure(caso_de_teste_didatico)
metrica_precisao.measure(caso_de_teste_didatico)
metrica_relev_contexto.measure(caso_de_teste_didatico)

print(f"\nRecall do Contexto: {metrica_recall.score}")
print(f"Precisão do Contexto: {metrica_precisao.score}")
print(f"Relevância Enxuta do Contexto: {metrica_relev_contexto.score}")

⚖️ Iniciando julgamento do Contexto/Retriever...



Recall do Contexto: 1.0
Precisão do Contexto: 1.0
Relevância Enxuta do Contexto: 1.0


---
## 🎓 Fase 5: Integração Langfuse + DeepEval (Observabilidade 360)
O poder verdadeiro existe quando você joga os *Scores* calculados acima de volta para a mesma nuvem que mediu a latência e o custo (O Trace!). Integrar DeepEval aos Traces do Langfuse.

Para isso, precisaríamos do ID do Trace rodado na Fase 3. Como eles rodam integrados via middleware num sistema de produção (na API local de callback), aqui mostramos a conexão visual conceitual.

In [5]:
from langfuse import Langfuse

langfuse = Langfuse()

# Didático: Como adicionar Scores ao Langfuse a partir das métricas rodando assíncronas
def vincular_scores_ao_trace(trace_id: str):
    
    langfuse.score(
        trace_id=trace_id,
        name="Answer Relevancy",
        value=metrica_relevancia.score,
        comment=metrica_relevancia.reason
    )
    
    langfuse.score(
        trace_id=trace_id,
        name="Faithfulness",
        value=metrica_fidelidade.score,
        comment=metrica_fidelidade.reason
    )
    
    langfuse.score(
        trace_id=trace_id,
        name="Contextual Recall",
        value=metrica_recall.score,
        comment=metrica_recall.reason
    )
    
    langfuse.flush()
    print("🟢 Métricas injetadas no Langfuse Cloud com sucesso!")

# Exemplo de invocação isolada simulando trace prévio: vincular_scores_ao_trace("id-do-trace-anterior-55b2...")

## 💥 Casos de Falha Extremos (Edge Cases)

Vamos mostrar didaticamente o porquê destas métricas serem essenciais. O que acontece se a nossa Vector DB trouxer uma evidência inútil (Retriever falho), mas o LLM, como é "esperto", responder a pergunta puxando da memória sistêmica dele (Aura de Inteligência)?

A resposta dele seria ÓTIMA. Mas a fidelidade seria PÉSSIMA (Alucinada em relação ao contexto). Sem essa separação métrica, a empresa acreditaria que o Retriever de PDFs é bom, quando na verdade está inútil.


In [6]:
teste_extremo = LLMTestCase(
    input="Qual a capital da França?",
    actual_output="A capital da França é Paris, uma cidade icônica europeia.", # Perfeito
    expected_output="A capital da frança é Paris.", # Perfeito
    retrieval_context=["O sistema solar é composto por 8 planetas e o Sol no centro."] # O RETRIEVER FALHOU FEIO NAS BUSCAS ESTELARES DA EMPRESA
)

# Note que a RELEVÂNCIA continua alta, porque a Resposta e a Pergunta conversam maravilhosamente bem.
metrica_relevancia.measure(teste_extremo)
print(f"Relevância: {metrica_relevancia.score}")

# Note que a FIDELIDADE cai massivamente para ZERO. Afinal o LLM não usou o contexto para responder, ele o ignorou e confiou em sua sabedoria local.
metrica_fidelidade.measure(teste_extremo)
print(f"Fidelidade: {metrica_fidelidade.score} -> ALUCINAÇÃO CONFIRMADA. O LLM DESPREZOU OS DOCUMENTOS EMPRESARIAIS!")

# Note que o RECALL é ZERO. Os documentos resgatados não cobriram absolutamente nada da Resposta Ideal esperada.
metrica_recall.measure(teste_extremo)
print(f"Recall: {metrica_recall.score} -> BANCO DE VETORES INOPERANTE PARA ESTAS QUESTÕES")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Relevância: 1.0


Fidelidade: 1.0 -> ALUCINAÇÃO CONFIRMADA. O LLM DESPREZOU OS DOCUMENTOS EMPRESARIAIS!


Recall: 0.0 -> BANCO DE VETORES INOPERANTE PARA ESTAS QUESTÕES
